|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the streaming detokenizer<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
print('vocab', tok.vocab_size)

Write the streaming detokenizer.

No GPU, no tensors, and no interesting arithmetic. This stage is boring and it
is the source of most user-visible bugs in real servers, which is a fair
trade for one afternoon.

# Exercise 1: emit the difference, not the token

A character can be split across two tokens, so decoding one token at a time
produces replacement characters. Decode the prefix instead and emit whatever
is new.

In [ ]:
class Incremental:
  def __init__(self, tokenizer):
    self.tok = tokenizer; self.ids = []; self.emitted = 0

  def push(self, token_id):
    """Return the NEW text this token produced, possibly ''."""
    self.ids.append(token_id)

    # decode the whole prefix, not the token
    text = 

    # a trailing replacement character means the last character is not
    # finished. Emit nothing and wait for the next token.
    

    # emit only what is new
    out = 
    self.emitted = 
    return out

  def flush(self):
    """The stream ended. Emit whatever is left, broken or not.

    Without this a reply that ends mid-character loses its last
    character forever, and the fuzz test below will find it."""
    

ids = tok('hello world', add_special_tokens=False).input_ids
d = Incremental(tok)
print([d.push(i) for i in ids])

# Exercise 2: prove it with a fuzz test

The contract is an equality. Test it that way.

In [ ]:
import random

# the contract is an equality: streamed text == batch-decoded text.
# Prove it on a lot of random token sequences, not on examples.
rng = random.Random(0)
fails = 0
for _ in range(500):
  ids = [rng.randrange(tok.vocab_size) for _ in range(rng.randint(1, 40))]
  
print(f'{fails} failures in 500 random sequences')

hard = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466',
        '\U0001F3F3\ufe0f\u200d\U0001F308', 'caf\u00e9', '\u65e5\u672c\u8a9e',
        '\uc548\ub155\ud558\uc138\uc694']
for t in hard:
  ids = tok(t, add_special_tokens=False).input_ids
  
  print(f'{ok}  {t!r}')

# Exercise 3: stop strings that straddle

The model emits ` EN` then `D`. Neither token contains `END`.

In [ ]:
def stream_until_stop(ids, stop):
  """Emit text until `stop` appears. Never emit a prefix of `stop`.

  The stop string can straddle a token boundary, so you cannot look at
  tokens. And you cannot emit eagerly: 'EN' must not flash up before you
  find out it was the start of 'END'.
  """
  d, out, hold = Incremental(tok), [], ''
  for i in ids:
    hold += d.push(i)

    # has the stop string appeared? emit up to it and stop.
    

    # otherwise hold back any suffix of `hold` that is a prefix of `stop`
    keep = 0
    
    out.append(hold[:len(hold)-keep] if keep else hold)
    hold = hold[len(hold)-keep:] if keep else ''
  return ''.join(out) + hold + d.flush(), False

cases = [('Answer: yes. END OF LINE', 'END'),
         ('nothing to stop for', 'END'),
         ('the ENDING is near', 'END')]
for text, stop in cases:
  ids = tok(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(ids, stop)
  print(f'{text!r}\n  -> {emitted!r}  stopped={stopped}')

### Before you open the solution

1. Run the third case, `'the ENDING is near'`. Did it stop? Is that
   correct? Would a user agree?
2. Your detokenizer decodes the whole prefix every time, which is O(n)
   per token and O(n^2) for a reply. At what output length does that
   start to matter, and what would you keep instead of all the ids?
3. What happens if you emit eagerly and check for the stop string
   afterwards? Describe what the user sees.